# MIT-BIH Dataset Creation for ECG Classification

This notebook processes MIT-BIH Arrhythmia Database files to create a standardized dataset for training ECG classification models.

## What This Notebook Does:
1. **Load all MIT-BIH records** from `/kaggle/input/mitbih-database/` (excluding 119.csv for testing)
2. **Extract individual beats** around R-peaks using traditional signal processing
3. **Resample each beat** to fixed 188 samples for CNN1D compatibility
4. **Include RR interval** as an additional feature (beat duration)
5. **Create binary labels**: N = Normal (0), everything else = Abnormal (1)
6. **Output**: CSV file with 189 columns (188 samples + RR interval) and label

## Beat Extraction Algorithm (Non-AI):
- Uses R-peak annotations from MIT-BIH
- Extracts a window centered on each R-peak
- Resamples to 188 samples using scipy.signal.resample
- Calculates RR interval from consecutive R-peaks

## STEP 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.signal import resample
from glob import glob
import warnings
warnings.filterwarnings('ignore')

# Configuration
BEAT_LENGTH = 188  # Fixed number of samples per beat for CNN1D
SAMPLING_RATE = 360  # MIT-BIH sampling rate (Hz)
EXCLUDE_RECORDS = ['119']  # Reserve for testing

print('Libraries imported successfully')
print(f'Target beat length: {BEAT_LENGTH} samples')
print(f'MIT-BIH sampling rate: {SAMPLING_RATE} Hz')
print(f'Excluded records (for testing): {EXCLUDE_RECORDS}')

## STEP 2: Define Data Paths

Adjust paths based on your environment (Kaggle or local).

In [ ]:
# Try Kaggle path first, then local
KAGGLE_PATH = '/kaggle/input/mitbih-database/'
LOCAL_PATH = '../deploy/sample/'  # Fallback for local testing

if os.path.exists(KAGGLE_PATH):
    DATA_PATH = KAGGLE_PATH
    print(f'Using Kaggle path: {DATA_PATH}')
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
    print(f'Using local path: {DATA_PATH}')
else:
    raise FileNotFoundError('No valid data path found!')

# List all CSV files (ECG signals)
signal_files = sorted(glob(os.path.join(DATA_PATH, '*.csv')))
print(f'\nFound {len(signal_files)} signal files')

# Filter out excluded records
signal_files = [f for f in signal_files if not any(excl in os.path.basename(f) for excl in EXCLUDE_RECORDS)]
print(f'After excluding {EXCLUDE_RECORDS}: {len(signal_files)} files')

## STEP 3: Parse Annotation File Format

MIT-BIH annotation files have the format:
```
Time   Sample #  Type  Sub Chan  Num	Aux
0:00.000	  0	+	0	0	(N
0:00.239	 86	N	0	0	
```

In [ ]:
def parse_annotations(annotation_file):
    """
    Parse MIT-BIH annotation file.
    
    Returns:
        List of (sample_index, beat_type) tuples
    """
    annotations = []
    
    try:
        with open(annotation_file, 'r') as f:
            lines = f.readlines()
        
        # Skip header line
        for line in lines[1:]:
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    sample_idx = int(parts[1])
                    beat_type = parts[2]
                    
                    # Skip non-beat annotations (e.g., rhythm markers)
                    # Valid beat types include: N, L, R, B, A, a, J, S, V, r, F, e, j, n, E, /, f, Q, ?
                    if beat_type not in ['+', '~', '|', 's', 'T', '*', 'D', '=', '"', '@']:
                        annotations.append((sample_idx, beat_type))
                except (ValueError, IndexError):
                    continue
    except Exception as e:
        print(f'Error parsing {annotation_file}: {e}')
        
    return annotations

# Test with sample file
test_ann_file = os.path.join(DATA_PATH, '100annotations.txt')
if os.path.exists(test_ann_file):
    test_annotations = parse_annotations(test_ann_file)
    print(f'Test parse: {len(test_annotations)} annotations from 100annotations.txt')
    print(f'First 5 annotations: {test_annotations[:5]}')
    
    # Count beat types
    beat_types = {}
    for _, bt in test_annotations:
        beat_types[bt] = beat_types.get(bt, 0) + 1
    print(f'Beat type distribution: {beat_types}')

## STEP 4: Beat Extraction and Resampling

Extract individual beats using:
1. **R-peak location** from annotations
2. **Fixed window** around R-peak (proportional before/after)
3. **Resampling** to 188 samples using scipy

In [ ]:
def extract_beat(signal, r_peak_idx, prev_r_peak_idx, next_r_peak_idx, target_length=188):
    """
    Extract and resample a single beat.
    
    Args:
        signal: Full ECG signal array
        r_peak_idx: Index of current R-peak
        prev_r_peak_idx: Index of previous R-peak (for RR interval and window start)
        next_r_peak_idx: Index of next R-peak (for window end)
        target_length: Number of samples to resample to
        
    Returns:
        Tuple of (resampled_beat, rr_interval_normalized)
    """
    signal_length = len(signal)
    
    # Calculate window boundaries
    # Use 30% of RR interval before R-peak, 70% after (typical ECG morphology)
    rr_before = r_peak_idx - prev_r_peak_idx
    rr_after = next_r_peak_idx - r_peak_idx
    
    # Window: 30% of previous RR before, 70% of next RR after R-peak
    window_start = max(0, r_peak_idx - int(0.3 * rr_before))
    window_end = min(signal_length, r_peak_idx + int(0.7 * rr_after))
    
    # Extract beat segment
    beat = signal[window_start:window_end]
    
    # Handle edge cases (too short)
    if len(beat) < 10:
        return None, None
    
    # Resample to target length
    resampled_beat = resample(beat, target_length)
    
    # Calculate RR interval (normalized by sampling rate)
    rr_interval = (rr_before + rr_after) / 2.0  # Average RR interval
    rr_interval_normalized = rr_interval / SAMPLING_RATE  # Convert to seconds
    
    return resampled_beat, rr_interval_normalized


def process_record(signal_file, annotation_file):
    """
    Process a single MIT-BIH record.
    
    Returns:
        List of (beat_samples, rr_interval, label) tuples
    """
    # Load signal (use first column - typically MLII lead)
    try:
        signal_df = pd.read_csv(signal_file, header=None)
        if signal_df.shape[1] > 1:
            # Multi-column: first column is sample index, second is signal
            if signal_df.iloc[0, 0] == 0 or signal_df.iloc[0, 0] == "'Sample #0":
                signal = signal_df.iloc[:, 1].values.astype(float)
            else:
                signal = signal_df.iloc[:, 0].values.astype(float)
        else:
            signal = signal_df.iloc[:, 0].values.astype(float)
    except Exception as e:
        print(f'Error loading {signal_file}: {e}')
        return []
    
    # Parse annotations
    annotations = parse_annotations(annotation_file)
    if len(annotations) < 3:
        print(f'Too few annotations in {annotation_file}')
        return []
    
    beats_data = []
    
    # Process each beat (skip first and last for RR calculation)
    for i in range(1, len(annotations) - 1):
        prev_sample, _ = annotations[i - 1]
        curr_sample, beat_type = annotations[i]
        next_sample, _ = annotations[i + 1]
        
        # Extract beat
        beat, rr = extract_beat(signal, curr_sample, prev_sample, next_sample, BEAT_LENGTH)
        
        if beat is None:
            continue
        
        # Create binary label: N = Normal (0), everything else = Abnormal (1)
        label = 0 if beat_type == 'N' else 1
        
        beats_data.append((beat, rr, label, beat_type))
    
    return beats_data


print('Beat extraction functions defined')

## STEP 5: Process All Records

In [ ]:
all_beats = []
record_stats = []

print('Processing MIT-BIH records...')
print('=' * 70)

for signal_file in signal_files:
    # Get record name (e.g., '100' from '100.csv')
    record_name = os.path.basename(signal_file).replace('.csv', '')
    
    # Skip excluded records
    if record_name in EXCLUDE_RECORDS:
        print(f'Skipping {record_name} (reserved for testing)')
        continue
    
    # Find annotation file
    annotation_file = os.path.join(DATA_PATH, f'{record_name}annotations.txt')
    
    if not os.path.exists(annotation_file):
        # Try alternative naming
        annotation_file = os.path.join(DATA_PATH, f'a{record_name}annotations.txt')
        if not os.path.exists(annotation_file):
            print(f'No annotation file for {record_name}, skipping')
            continue
    
    # Process record
    beats = process_record(signal_file, annotation_file)
    
    if len(beats) > 0:
        n_normal = sum(1 for b in beats if b[2] == 0)
        n_abnormal = sum(1 for b in beats if b[2] == 1)
        
        print(f'Record {record_name}: {len(beats)} beats (Normal: {n_normal}, Abnormal: {n_abnormal})')
        
        all_beats.extend(beats)
        record_stats.append({
            'record': record_name,
            'total': len(beats),
            'normal': n_normal,
            'abnormal': n_abnormal
        })

print('\n' + '=' * 70)
print(f'Total beats extracted: {len(all_beats)}')
print(f'Normal beats: {sum(1 for b in all_beats if b[2] == 0)}')
print(f'Abnormal beats: {sum(1 for b in all_beats if b[2] == 1)}')

## STEP 6: Analyze Beat Type Distribution

In [ ]:
# Analyze beat types
beat_type_counts = {}
for _, _, label, beat_type in all_beats:
    key = f'{beat_type} ({"Normal" if label == 0 else "Abnormal"})'
    beat_type_counts[key] = beat_type_counts.get(key, 0) + 1

print('Beat Type Distribution:')
print('=' * 50)
for bt, count in sorted(beat_type_counts.items(), key=lambda x: -x[1]):
    print(f'{bt:30s}: {count:6d}')

# Print record statistics
print('\n' + '=' * 70)
print('Per-Record Statistics:')
print('=' * 70)
stats_df = pd.DataFrame(record_stats)
if len(stats_df) > 0:
    print(stats_df.to_string(index=False))

## STEP 7: Create Dataset DataFrame

In [ ]:
# Create column names
# 188 sample columns + 1 RR interval column + 1 label column
columns = [f'sample_{i}' for i in range(BEAT_LENGTH)] + ['rr_interval', 'label']

# Build dataset
data_rows = []
for beat, rr, label, _ in all_beats:
    row = list(beat) + [rr, label]
    data_rows.append(row)

# Create DataFrame
df = pd.DataFrame(data_rows, columns=columns)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {len(df.columns)} (188 samples + 1 RR interval + 1 label)')
print(f'\nFirst few rows:')
print(df.head())

print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nLabel percentages:')
print(df['label'].value_counts(normalize=True) * 100)

## STEP 8: Data Quality Checks

In [ ]:
print('Data Quality Checks')
print('=' * 50)

# Check for NaN values
nan_count = df.isna().sum().sum()
print(f'NaN values: {nan_count}')

# Check for infinite values
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f'Infinite values: {inf_count}')

# Check RR interval range
rr_stats = df['rr_interval'].describe()
print(f'\nRR Interval Statistics (seconds):')
print(rr_stats)

# Check sample value range
sample_cols = [c for c in df.columns if c.startswith('sample_')]
sample_min = df[sample_cols].min().min()
sample_max = df[sample_cols].max().max()
sample_mean = df[sample_cols].mean().mean()
print(f'\nSample Value Range:')
print(f'  Min: {sample_min:.2f}')
print(f'  Max: {sample_max:.2f}')
print(f'  Mean: {sample_mean:.2f}')

# Remove any rows with NaN or Inf
original_len = len(df)
df = df.dropna()
df = df[~np.isinf(df.select_dtypes(include=[np.number])).any(axis=1)]
if len(df) < original_len:
    print(f'\nRemoved {original_len - len(df)} invalid rows')
    print(f'Final dataset size: {len(df)}')

## STEP 9: Shuffle and Save Dataset

In [ ]:
# Shuffle the dataset
RANDOM_STATE = 42
df_shuffled = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Dataset shuffled with random_state={RANDOM_STATE}')
print(f'First 5 labels after shuffle: {df_shuffled["label"].head().tolist()}')

# Save to CSV
output_path = 'mitbih_beats_dataset.csv'
df_shuffled.to_csv(output_path, index=False)

print(f'\nDataset saved to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB')

# Also save without RR interval (for compatibility with original 188-column format)
output_path_no_rr = 'mitbih_beats_dataset_no_rr.csv'
sample_cols = [c for c in df_shuffled.columns if c.startswith('sample_')]
df_no_rr = df_shuffled[sample_cols + ['label']]
df_no_rr.columns = list(range(188)) + ['label']  # Match original format
df_no_rr.to_csv(output_path_no_rr, index=False)

print(f'\nCompatibility dataset saved to: {output_path_no_rr}')
print(f'File size: {os.path.getsize(output_path_no_rr) / 1024 / 1024:.2f} MB')

## STEP 10: Visualize Sample Beats

In [ ]:
import matplotlib.pyplot as plt

# Sample normal and abnormal beats
normal_beats = df_shuffled[df_shuffled['label'] == 0].head(5)
abnormal_beats = df_shuffled[df_shuffled['label'] == 1].head(5)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

sample_cols = [c for c in df_shuffled.columns if c.startswith('sample_')]

for i in range(5):
    # Normal beats (top row)
    if i < len(normal_beats):
        axes[0, i].plot(normal_beats.iloc[i][sample_cols].values, color='green', linewidth=1)
        rr = normal_beats.iloc[i]['rr_interval']
        axes[0, i].set_title(f'Normal (RR={rr:.3f}s)')
    axes[0, i].set_xlabel('Sample')
    axes[0, i].set_ylabel('Amplitude')
    axes[0, i].grid(True, alpha=0.3)
    
    # Abnormal beats (bottom row)
    if i < len(abnormal_beats):
        axes[1, i].plot(abnormal_beats.iloc[i][sample_cols].values, color='red', linewidth=1)
        rr = abnormal_beats.iloc[i]['rr_interval']
        axes[1, i].set_title(f'Abnormal (RR={rr:.3f}s)')
    axes[1, i].set_xlabel('Sample')
    axes[1, i].set_ylabel('Amplitude')
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Sample ECG Beats (Top: Normal, Bottom: Abnormal)', fontsize=14)
plt.tight_layout()
plt.savefig('sample_beats_visualization.png', dpi=150)
plt.show()

print('Visualization saved to: sample_beats_visualization.png')

## STEP 11: Summary Statistics

In [ ]:
print('=' * 70)
print('DATASET CREATION SUMMARY')
print('=' * 70)

print(f'\n1. INPUT:')
print(f'   - Source: MIT-BIH Arrhythmia Database')
print(f'   - Records processed: {len(record_stats)}')
print(f'   - Records excluded: {EXCLUDE_RECORDS}')

print(f'\n2. PROCESSING:')
print(f'   - Beat extraction: R-peak centered, 30%/70% window')
print(f'   - Resampling: {BEAT_LENGTH} samples per beat')
print(f'   - RR interval: Normalized to seconds')

print(f'\n3. OUTPUT:')
print(f'   - Total beats: {len(df_shuffled)}')
print(f'   - Normal beats: {len(df_shuffled[df_shuffled["label"] == 0])} ({len(df_shuffled[df_shuffled["label"] == 0])/len(df_shuffled)*100:.1f}%)')
print(f'   - Abnormal beats: {len(df_shuffled[df_shuffled["label"] == 1])} ({len(df_shuffled[df_shuffled["label"] == 1])/len(df_shuffled)*100:.1f}%)')

print(f'\n4. FILES CREATED:')
print(f'   - {output_path} (with RR interval, 190 columns)')
print(f'   - {output_path_no_rr} (without RR, 189 columns)')

print(f'\n5. LABEL ENCODING:')
print(f'   - 0 = Normal (N)')
print(f'   - 1 = Abnormal (all other beat types)')

print(f'\n6. NEXT STEPS:')
print(f'   - Use mitbih_beats_dataset.csv for v6 training')
print(f'   - Test model on excluded record 119')
print(f'   - Expected realistic accuracy: 85-95%')

print('\n' + '=' * 70)
print('Dataset creation complete!')
print('=' * 70)